# CDI Layer 2 and Layer 3

Use the **compute** kernel. After repository code changes, restart the kernel once. Put `OPENAI_API_KEY` in the repository `.env`, replace `FACT_SHEET` below with an absolute path to a structured Markdown fact sheet, and run this cell. It creates a resumable eight-mission Layer 2 run and shows the deterministic report plus one mission for inspection.

In [ ]:
from __future__ import annotations

import asyncio
import json
import os
from pathlib import Path

from IPython.display import JSON, Markdown, clear_output, display

from ML.deep_research.layer2.cli import load_dotenv_key, run_all as run_layer2
from ML.deep_research.layer2.create_run import create_run as create_layer2_run
from ML.deep_research.layer2.fs import load_json, read_text
from ML.deep_research.layer2.settings import PLANNER_PATH, RUNS_DIR

FACT_SHEET = Path(r"fact_sheet.md")
PREVIEW_MISSION = 0

load_dotenv_key()
if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError("Add OPENAI_API_KEY to the repository .env before running Layer 2.")

L2_RUN = create_layer2_run(FACT_SHEET, PLANNER_PATH, RUNS_DIR)
print(f"Layer 2 run created: {L2_RUN}")
print(f"Resume if interrupted: .\\run.ps1 -Resume '{L2_RUN}'")
L2_CHECKS = await asyncio.to_thread(run_layer2, L2_RUN)

l2_record = load_json(L2_RUN / "run.json")
mission_files = sorted((L2_RUN / "missions").glob("*.json"))
display(Markdown(read_text(L2_RUN / "check_report.md")))
display(Markdown("### Recorded Layer 2 usage"))
display(JSON(data=l2_record.get("usage", {}), expanded=True))
print(f"Mission files: {len(mission_files)}")
for path in mission_files:
    print(f"- {path.name}")
if mission_files:
    preview = mission_files[PREVIEW_MISSION]
    display(Markdown(f"### Mission preview: `{preview.name}`"))
    display(JSON(data=load_json(preview), expanded=False))


## Layer 3 — live online research

Run this only after Layer 2 completes. **This cell confirms that the input is public or invented, sends research queries to external services, and consumes model/web-search usage.** Layer 3 currently uses high reasoning; there is no separate test-mode command. Eight researchers run in parallel, followed by one review, one optional clarification batch, and synthesis. The cell shows every published domain report, the review, usage, and the final property answer.

In [2]:
from __future__ import annotations

import asyncio
import json
import os
from pathlib import Path

from IPython.display import JSON, Markdown, clear_output, display

from ML.deep_research.layer2.cli import load_dotenv_key
from ML.deep_research.layer2.fs import load_json, read_text
from ML.deep_research.layer3.cli import run_all as run_layer3
from ML.deep_research.layer3.pipeline.create_run import create_run as create_layer3_run
from ML.deep_research.layer3.settings import RUNS_DIR as LAYER3_RUNS_DIR, SCHEMA_VERSION
from ML.deep_research.layer3.usage import summarize_usage

PUBLIC_INPUT_CONFIRMED = True
load_dotenv_key()
if not globals().get("L2_RUN"):
    for candidate in sorted(
        LAYER3_RUNS_DIR.glob("L2_*"), key=lambda path: path.stat().st_mtime, reverse=True
    ):
        record = load_json(candidate / "run.json")
        checks = record.get("checks", {})
        if checks.get("run") and checks.get("passed") == checks.get("run"):
            L2_RUN = candidate
            break
    else:
        raise RuntimeError("No completed Layer 2 run was found under runs/.")
L2_RUN = Path(L2_RUN)
print(f"Using existing Layer 2 run: {L2_RUN}")

if not PUBLIC_INPUT_CONFIRMED:
    raise RuntimeError("Layer 3 requires explicit confirmation of public or invented input.")
if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError("Add OPENAI_API_KEY to the repository .env before running Layer 3.")

for candidate in sorted(
    LAYER3_RUNS_DIR.glob("L3_*"), key=lambda path: path.stat().st_mtime, reverse=True
):
    candidate_record = load_json(candidate / "run.json")
    source_path = candidate_record.get("source_l2", {}).get("path", "")
    if (
        candidate_record.get("schema_version") == SCHEMA_VERSION
        and source_path
        and Path(source_path).resolve() == L2_RUN.resolve()
    ):
        L3_RUN = candidate
        break
else:
    L3_RUN = create_layer3_run(
        L2_RUN, LAYER3_RUNS_DIR, public_input_confirmed=PUBLIC_INPUT_CONFIRMED
    )

l3_before = load_json(L3_RUN / "run.json")
execution = l3_before.get("execution", {})
records = [
    *execution.get("domains", {}).values(),
    execution.get("review", {}),
    execution.get("final", {}),
    *(((execution.get("clarification") or {}).get("domains", {})).values()),
]
retry_failed = any(item.get("status") == "failed" for item in records)
retry_flag = " -RetryFailed" if retry_failed else ""
resume_command = f".\\run.ps1 -ResumeL3 '{L3_RUN}'{retry_flag}"
print(f"Layer 3 run: {L3_RUN}")
print(f"Resume if interrupted: {resume_command}")
layer3_task = None
if l3_before.get("status") != "verified":
    layer3_task = asyncio.create_task(run_layer3(L3_RUN, retry_failed=retry_failed))
while layer3_task and not layer3_task.done():
    await asyncio.sleep(5)
    live = load_json(L3_RUN / "run.json")
    lines = read_text(L3_RUN / "usage.jsonl").splitlines()
    latest = json.loads(lines[-1]) if lines else {}
    running = [
        name for name, item in live.get("execution", {}).get("domains", {}).items()
        if item.get("status") == "running"
    ]
    partials = sorted((L3_RUN / "domains").glob("*.partial.md"))
    clear_output(wait=True)
    print(f"Layer 3 run: {L3_RUN}")
    print(f"Resume if interrupted: {resume_command}")
    print(f"Running domains: {len(running)}")
    for name in running:
        print(f"- {name}")
    print(f"Progressive reports: {len(partials)}/8")
    print("Usage:", summarize_usage(L3_RUN))
    if latest:
        print("Last activity:", latest.get("timestamp"), latest.get("actor"), latest.get("phase"), latest.get("detail", ""))
L3_CHECKS = await layer3_task if layer3_task else []
clear_output(wait=True)

l3_record = load_json(L3_RUN / "run.json")
domain_files = sorted(
    path for path in (L3_RUN / "domains").glob("*.md")
    if not path.name.endswith(".partial.md")
)
partial_files = sorted((L3_RUN / "domains").glob("*.partial.md"))
review = L3_RUN / "review" / "final_review.md"
final_answer = L3_RUN / "research" / "final.md"
display(Markdown(read_text(L3_RUN / "check_report.md")))
display(Markdown("### Recorded Layer 3 usage"))
display(JSON(data=l3_record.get("usage", {}), expanded=True))
print(f"Domain reports: {len(domain_files)}")
for path in domain_files:
    print(f"- {path.name}")
print(f"Progressive report logs retained: {len(partial_files)}")
display(Markdown("### Final review"))
display(Markdown(read_text(review) if review.is_file() else "Not produced; see the check report above."))
display(Markdown("### Property synthesis"))
display(Markdown(read_text(final_answer) if final_answer.is_file() else "Not produced; see the check report above."))


# Run L3_20260821_29a2 — Layer 3 check report

14 checks · 13 passed · 1 failed

## Checks

- [x] 1. The recorded Layer 2 result passed every check
- [x] 2. Skill and prompt snapshots match their recorded hashes
- [x] 3. Run schema, harness, model policy, provider, and public-input record are valid
- [x] 4. Eight domains, one review, optional single clarification, and synthesis are complete
- [x] 5. All 8 published and progressive domain reports are non-empty — expected=8 published and 8 partial
- [x] 6. The adversarial review is non-empty
- [x] 7. The final property answer is non-empty
- [x] 8. Every retained raw source hashes to its filename
- [x] 9. Source index paths and hashes resolve
- [x] 10. Query records are unique and structurally valid
- [x] 11. Citation records are unique and exact quotations revalidate
- [ ] 12. Every emitted citation marker resolves to verified evidence
- [x] 13. An answered property has a verified final-answer citation
- [x] 14. Usage and progress events are attributable, timestamped, and unique


### Recorded Layer 3 usage

<IPython.core.display.JSON object>

Domain reports: 8
- asset-integrity-systems-and-operational-resilience.md
- energy-carbon-and-transition.md
- external-dependencies-geopolitics-trade-and-supply-chains.md
- finance-debt-and-macro-transmission.md
- ground-physical-climate-and-insurability.md
- location-demand-market-valuation-and-exit.md
- occupier-lease-income-and-counterparty-economics.md
- rights-public-law-and-ownership-governance.md
Progressive report logs retained: 8


### Final review

# Comprehensive property review

## Decision-level conclusions

**Decision: proceed only conditionally; do not approve unconditional acquisition, financing, or value reliance at this stage.** The asset is credibly identified as the Amtsgericht Bad Homburg v.d. Höhe courthouse complex at Auf der Steinkaut 10–12 and is in active courthouse use. The evidence supports a current stated rent of EUR 78,255.17 per month, or EUR 939,062.04 annualised, and a real prospective garage-renewal project with tender prices clustered around EUR 0.70–0.72 million net. Those positives are not yet sufficient for a clean investment case.

The principal decision blockers are:

1. **Title and legal envelope:** current parcels, registered rights, easements, Baulasten, charge releases and the applicable planning regime are not verified. The data-room statement that the site is governed by §34 BauGB and lies outside a Bebauungsplan conflicts with the City's public listing of Bebauungsplan No. 117, which includes Auf der Steinkaut in its title.
2. **Income durability:** the face-rent history is documented, but the complete executed lease chain, legal tenant and payer, term, break rights, recoveries, payment history, security and enforceability of indexation are not established. Public courthouse occupation is not proof of a State-backed contractual cash flow.
3. **Current technical and life-safety condition:** the inspection trail is extensive but date-specific. Current fire-safety, HPPVO, escape-plan, structural, drainage, water, mechanical and electrical compliance cannot be demonstrated. The available record includes outdated escape/rescue plans, missing inspection reports and missing evidence of remediation.
4. **Capital and operating obligations:** the garage works are not evidenced as awarded. The EUR 702,668.06 net Chemicon recommendation is a tender-stage planning amount, not a committed liability. The safer current planning allowance remains EUR 765,000 net, before VAT/recoverability, design fees, testing, contingency, escalation and disruption. The FMC maintenance arrangement does not transfer on sale, creating an immediate Opex and service-continuity gap.
5. **Financing and insurability:** current debt, lender, balance, maturity, pricing, covenants, release status, liquidity and insurance cover are unknown. LTV, DSCR, refinancing risk, net sale proceeds and insurance recovery cannot be calculated reliably.

The prudent base case is therefore: **current face rent only; no value for unverified recoveries, security, further indexation, parking income or residual/development value; a specific garage-capex downside allowance; an unquantified life-safety, drainage, energy-transition and maintenance-transition reserve; and no reliance on the historic charge being released or on the building being fully financeable.**

The decision should improve materially if current title and planning records, a complete lease and clean receipts, current compliance certificates, a funded and contracted works programme, replacement maintenance cover, current debt/release evidence and binding insurance indications are delivered. A material title/planning defect, short or breakable lease, disputed rent, current life-safety failure, unreleased security, unavailable insurance or materially broader works scope should trigger repricing, escrow/indemnity, a works exclusion or rejection.

## Contradictions and the evidence supporting each reading

### Parcel identity and registered land

- **Reading A — two current parcels:** the asset-integrity report identifies parcels 122/11 and 123/1, totalling 6,361 m², and treats them as the current physical scope.
- **Reading B — unresolved parcel history:** the rights report records references to 122/9, 122/1, 122/11, 123/1 and 123/a and states that the owner, exact parcels, continuity of parcel changes and current registered burdens cannot be concluded without a certified Grundbuch and cadastral history.
- **Assessment:** Reading A is a reasonable working physical baseline, not title proof. The historic identifiers could reflect parcel changes or document errors. The exact acquired land and whether all buildings, access, parking and drainage lie within it remain a hard closing condition.

### Planning regime: §34 BauGB versus Bebauungsplan No. 117

- **Reading A — unplanned inner area:** the property-file planning material reportedly states that the site is an Innenbereich case under §34 BauGB and outside a Bebauungsplan.
- **Reading B — a plan may apply:** the City publicly lists Bebauungsplan No. 117, titled “Friesenstraße, Seedammweg, Auf der Steinkaut, Am Schützbrett, Frankfurter Landstraße,” and states that only the executed and properly published original plan is legally binding.
- **Assessment:** The plan title alone does not prove that parcels 122/11 and 123/1 fall within the plan. Conversely, the data-room §34 statement cannot be relied upon. The executed plan, official parcel overlay, amendments and written planning confirmation are required. Until then, expansion, conversion, alternative-use and residual-land value should be excluded.

### Water-protection designations

- **Reading A — healing-spring protection:** official geodata identifies HQS Bad Homburg, WSG-ID 434-060, as a designated Heilquellenschutzgebiet in quantitative protection Zone D.
- **Reading B — drinking-water designation:** other official material identifies Pfingstborn 1+2, WSG-ID 434-002, as Zone IIIA, while property/planning material refers to Pfingstborn II/IIIB.
- **Assessment:** These may be separate overlapping designations rather than a direct contradiction. They must not be harmonised by assumption. The applicable zone or zones, ordinances and parcel intersection must be established for both current parcels and for garage, drainage, dewatering, drilling and heating works.

### Parking provision and compensation

- **Reading A — 89 spaces with later coherent split:** the 1988 submission records 89 spaces as approximately 55 underground plus 34 outdoor.
- **Reading B — earlier inconsistent allocation:** the 1985 material also refers to 89 required spaces but describes approximately 58 underground plus 39 visitor spaces, totalling 97. A historic replacement-payment attachment refers to DM 213,000 before a stated cap and DM 200,000 for ten spaces.
- **Assessment:** The later arithmetic is coherent, but neither allocation is proven to govern today. The operative approval, current count, accessibility/fire-access position, easement scope, compensation agreement and payment status remain unresolved. Do not recognise a current parking shortfall, parking income or replacement liability without current records, but do not assume the issue is closed.

### Drainage capacity

- **Reading A — historic design basis:** the 1988 material records domestic wastewater of 5.81 l/s and total rainwater of 33.52 l/s.
- **Reading B — inconsistent calculation:** another page records rainwater as 33.62 l/s, a 0.10 l/s difference, and a drainage-canal easement is associated with a historic parcel 122/9.
- **Assessment:** The discrepancy does not prove inadequate capacity, but it prevents reliance on the historic calculation. Current connection approval, hydraulic capacity, backflow protection, maintenance responsibility, easement access and climate/design basis must be confirmed before intrusive works or any conclusion on operational resilience.

### Registered charge and debt

- **Reading A — large nominal charge:** one record prints a Eurohypo charge of EUR 595,375,000 with interest and ancillary benefits.
- **Reading B — likely decimal/document issue:** a release entry prints EUR 595,375. The reports correctly state that neither amount can be treated as current debt or exposure.
- **Assessment:** The discrepancy could reflect a transcription error, a nominal security amount, a release/assignment issue or cross-collateralisation. A current land-register extract, notarial instruments, lender/servicer confirmation and authenticated release or assignment are required. The registered charge does not disclose the loan balance, pricing, maturity or continuing debt.

### Fire-safety status

- **Reading A — historic requirements and dated observations:** the records establish a historic requirement for at least 2,400 l/min of fire water for two hours, recurring inspection intervals, and dated October/December 2025 fire-door and detector observations.
- **Reading B — current status unresolved:** the January 2026 table records outdated escape/rescue plans, unavailable HPPVO reports and missing LBIH evidence of closure, while some issues are said to have been remedied.
- **Assessment:** Historic requirements and dated observations are not proof of current failure, but neither are assertions of remediation proof of compliance. Current fire-water testing, fire-door/detector inspection, HPPVO reports, escape plans, authority acceptance and closure evidence are transaction conditions.

### Garage procurement

- **Reading A — competitive tender basis:** Chemicon EUR 702,668.06 net, Karrié EUR 704,397.29 net and Teixeira EUR 719,950.93 net are three close bids for the reported scope, against a EUR 765,000 net estimate.
- **Reading B — no evidenced construction award:** the February 2026 ptd plan+ appointment is for architectural services and LP 8 oversight for renewing the garage coating; it does not prove that Chemicon was appointed, that works began or that the scope is complete.
- **Assessment:** The bids support a cost reasonableness range and a planning allowance, not a fixed liability or saving. Obtain the signed award, controlled scope, exclusions, cost-to-complete, programme, approvals, warranties, bonds, insurance and acceptance records.

### Public occupation versus contractual income

- **Reading A — active public courthouse:** the official judiciary page confirms the Amtsgericht operates at the address and provides public access and parking information.
- **Reading B — tenant and payer unknown:** the lease reports do not establish whether the court, Land Hessen or another entity is the contractual tenant and payer, nor do they establish a guarantee, appropriation, term or payment history.
- **Assessment:** Public occupation is credible and operationally positive, but it cannot support a public-counterparty credit premium or long-duration assumption until the executed lease, tenant certificate, receipts and authority chain are reviewed.

### Lease-form analysis

- **Reading A — contractual §126 BGB control:** the 2020 amendment reportedly requires handwritten signatures by both parties on the same instrument for the lease and amendments.
- **Reading B — statutory transition complexity:** the rights report notes that the statutory treatment of commercial leases and amendments has changed, with transition rules distinguishing older leases from amendments agreed from 1 January 2025.
- **Assessment:** There is no conclusion that the lease is invalid or terminable. The complete signed chronology, incorporated schedules, side letters, notices and legal-form analysis are required. The contractual written-form covenant may create a stricter risk than the current statutory minimum.

### No-findings evidence versus clearance

- **Reading A — no current entries/no known findings:** the April 2026 ALTIS enquiry reportedly found no current entries or known contamination findings, and a 2006 certificate reported no UXO suspicion.
- **Reading B — no formal all-clear:** the reports correctly state that these are database or historical enquiry results, not intrusive clearance, current groundwater evidence or a warranty against contamination, UXO or archaeological finds. Wider-area archaeological records document Roman and Merovingian discoveries, but do not identify the subject parcels.
- **Assessment:** The evidence narrows the risk from demonstrated contamination or loss to unverified residual exposure. It should not be converted into either a negative finding or a positive clearance.

## Shared-source dependence and apparent agreement that is not independent confirmation

- The eight reports repeatedly rely on the same supplied property-file context: the lease amendments and rent figures, the parcel descriptions, the fire-safety status table, the historic drainage and parking records, the garage tender, the ptd appointment and the FMC non-transfer statement. Repetition across domains is not independent corroboration.
- The three garage bids are not three independent market comparables. They are bids to the same scope, likely based on common quantities, specifications, subcontractor markets and disposal channels. Their tight spread supports tender competition and arithmetic reasonableness only.
- The three rent increases are not independent income evidence. They are successive calculations within the same lease chain and may be arithmetically consistent while still being legally or operationally disputed.
- The active-courthouse conclusion is corroborated by an official judiciary page, but that public source confirms location and use, not contractual tenancy, payment liability, security or duration.
- The planning conflict is not resolved by multiple reports repeating it. The public City page and the property-file planning statement are distinct readings, but neither substitutes for the executed plan and parcel overlay.
- The water-risk conclusion has some independent official support from Geoportal, RP Darmstadt and city sources, but those sources establish regional or designated regimes, not the parcel-specific legal restrictions or loss probability.
- Market reports, municipal economic data and official transport information support the general attractiveness of Bad Homburg, but they do not independently confirm demand, value or liquidity for this specialised courthouse asset.
- Commerzbank's public history supports a lender-tracing route from Eurohypo, but it does not independently identify the current creditor, balance or discharge authority for this property.

## Duplicated financial effects and risks

The following effects should be consolidated rather than added once for every domain:

1. **Garage CapEx:** the EUR 702,668.06–EUR 765,000 net issue appears in asset integrity, occupier economics, external dependencies, finance and valuation. Count it once as a project cost, then add separately only evidenced professional fees, VAT, contingency, escalation and financing carry.
2. **Garage disruption:** parking displacement, restricted access, court disruption, potential abatement, claims and delay are one linked operational-loss scenario, not separate full deductions in lease, location, finance and asset models.
3. **Fire-safety exposure:** missing HPPVO records, escape-plan updates, fire-water testing, fire-door/detector observations, insurance concerns and possible closure are one life-safety/compliance pathway. Avoid layering multiple independent vacancy, capex and insurance haircuts unless each has a distinct quantified basis.
4. **FMC transition:** replacement maintenance procurement, mobilisation, overlap, open defects, statutory testing and service leakage are one initial Opex/continuity reserve. The same amount should not be deducted again as both maintenance Opex and a separate operational-resilience loss.
5. **Water, groundwater and drainage:** protection-area restrictions, dewatering, waterproofing, backflow, drainage capacity, garage coating and authority delay are connected design and programme risks. A single scoped technical contingency should be used, with separate additions only for distinct confirmed obligations.
6. **Parking:** current-count uncertainty, Hesse rights, replacement payment, access and garage-work disruption form one legal/operational issue. Do not apply both a permanent value haircut and a full income loss without establishing whether a lasting shortfall exists.
7. **Lease and public-counterparty uncertainty:** missing tenant identity, term, form, security, receipts and break rights are related income-durability risks. They should drive one contractual-income scenario set rather than multiple independent default, vacancy and yield penalties.
8. **Physical-risk insurance:** flood, heavy rain, groundwater, fire, business interruption and exclusions should be modelled through the uninsured-loss and deductible structure. Do not deduct the full gross loss and also assume full insurance exclusion unless coverage evidence supports that outcome.
9. **Energy transition:** fuel cost, CO₂ cost, automation compliance, heating replacement and stranded-asset risk are one transition pathway. Energy CapEx should be based on measured consumption, plant age and statutory applicability, not added as a generic reserve on top of every climate and Opex sensitivity.
10. **Value effects:** condition, planning, lease, financing and exit risks may already be reflected in an appropriately selected yield or valuation scenario. Avoid separately applying a yield expansion, vacancy deduction, CapEx deduction and specialised-asset haircut if the valuation already incorporates the same underlying risk.

## Cross-domain causal chains, handoffs and broken links

### Legal identity to works and value

Current parcel confirmation leads to the correct title, easement, Baulasten, planning overlay, water-zone and parking analysis. Those findings determine whether drainage, fire access, garage, roof-space and energy works are lawful and financeable. The link is broken at the certified Grundbuch, cadastral history, executed plan and current rights records.

### Lease execution to income and debt service

Complete signed lease chain and tenant identity lead to term, break, indexation, recoveries, obligations and payment rights. Receipts then establish effective income, which feeds NOI, DSCR, value and exit assumptions. The chain is broken by missing execution pages, schedules, notices, tenant certificate, receipts, arrears, service-charge data and security.

### Condition to operational continuity

Current technical survey leads to a defect register, urgency ranking, scope, programme and cost. Fire and drainage close-out then supports lawful occupation, insurance and uninterrupted court operations. The chain is broken by date-specific inspections, missing HPPVO/LBIH evidence, outdated escape plans, unreadable DWGs and missing current structural, hydraulic and system records.

### Garage design to completion and income leakage

The ptd LP 8 appointment should lead to a controlled specification, authority approvals, valid award, contractor mobilisation, phased access, testing and acceptance. Those steps determine parking availability, court disruption, abatement and cost-to-complete. The chain is broken between architectural appointment and evidenced construction award, and again at the missing programme, tenant protocol, approvals and funding.

### FMC expiry to Opex and resilience

Non-transfer of FMC requires a replacement provider or valid novation, plus records, warranties, open orders, statutory testing and emergency cover. Without that handoff, the buyer bears mobilisation and service-gap risk and may not know whether historic defects are closed. This is a day-one control, not merely an Opex forecasting issue.

### Energy baseline to transition CapEx

Bills, meter boundaries and plant data lead to a weather-normalised baseline. Plant capacity and age then determine whether automation, heating replacement or controls are required, while the local heat plan and water-protection regime determine feasible technology. The chain is broken by absent 36-month data, plant schedules, BMS information, floor area, heat-plan output and governing-law confirmation.

### Physical hazard to insurance and financing

Parcel-specific water, groundwater, radon, fire and other hazard evidence feeds mitigation design, insurer terms, deductibles, business interruption cover and lender conditions. The link is broken by the absence of current policy schedules, claims history, insurer survey and parcel-specific hazard interpretation.

### Debt and value to liquidity

Verified debt and security terms combine with verified NOI, value and CapEx timing to determine LTV, DSCR, covenant headroom, refinance gap and sale proceeds. The chain is broken by the unresolved charge, missing loan records, absent funding plan, unverified rent and unquantified works.

## Material unknowns and whether they should remain unknown

### Unknowns that should be resolved before commitment

- Current certified ownership, parcel boundaries, parcel history, Section II/III burdens, Baulasten and release of the Eurohypo charge.
- Executed and legally binding Bebauungsplan No. 117, parcel overlay, current planning confirmation, use permissions and approval status for existing and proposed works.
- Applicable healing-spring and drinking-water protection zones, ordinances and written authority position for garage, drainage, dewatering, discharge, drilling and heating works.
- Complete signed lease chain, tenant/payer identity, term, breaks, renewal, written-form compliance, indexation notices, payment receipts, arrears, security, recoveries, tax/VAT treatment and works/abatement rights.
- Current independent technical, structural, fire, HPPVO, escape-plan, mechanical, electrical, drainage, fire-water and water-system evidence.
- Current garage scope, award, programme, approvals, funding, contractor security, warranties and tenant operating protocol.
- Replacement maintenance arrangement, invoices, open work orders, asset/service matrix, warranties and statutory-test calendar.
- Current debt balance, lender/servicer, loan terms, maturity, hedge, covenants, guarantees, liquidity, release/assignment and lender consent.
- Current insurance schedule, claims history, exclusions, deductibles, business-interruption cover and works coverage.
- Energy bills, meter boundaries, certificates, plant capacity/age, controls and a dated legal opinion on heating and automation requirements.

These are not ordinary modelling assumptions. They directly control whether the asset can be acquired, occupied, financed, insured and exited.

### Unknowns that may reasonably remain unknown, provided they are priced honestly

- Exact future flood, heavy-rain, heat, wildfire, radon, earthquake or subsidence loss frequency. Public maps can improve screening but are unlikely to produce a reliable property-specific loss distribution without surveys, measurements, claims data and insurer underwriting.
- Future office rents, buyer depth, marketing period and exit yield for a specialised courthouse asset. Public submarket statistics can inform scenarios, but cannot create a verified specialist comparable set.
- Future indexation timing and future CPI. Model no further increase, delayed increase and valid full pass-through cases rather than treating future indexation as certain.
- Final local heat-network availability, carbon intensity and connection economics until the municipal heat-planning process produces an applicable result.
- Detailed geopolitical and supply-chain exposure for future works where supplier, subcontractor, origin, route and lead-time data are absent. This should remain a contingent procurement risk, not be converted into a fabricated probability.
- Archaeological findings and intrusive groundwater conditions before ground disturbance. Maintain a method, stop-work and contingency protocol rather than implying a current all-clear.

## Implications for income, Opex, CapEx, value, financing and exit

### Income

Use EUR 939,062.04 only as annualised stated face rent from January 2026. Do not call it NOI. Do not recognise recoveries, parking income, security, arrears recovery or further indexation until the lease, receipts, area schedule, service-charge records and counterparty are verified. Model a downside at the last undisputed rent and include delayed or disputed indexation. Public occupation supports continuity but not a contractual State guarantee.

### Opex

Existing maintenance Opex is not transferable through FMC and cannot be assumed to continue at the seller's cost or service level. Budget replacement procurement, mobilisation, overlap and statutory testing from completion. Add verified owner leakage for insurance, taxes, utilities, CO₂ costs and non-recoverable services only after the lease and accounts are reconciled. Energy Opex remains unquantified pending consumption data.

### CapEx

Carry at least EUR 702,668 net as the lowest evidenced garage tender and retain EUR 765,000 net as the safer pre-award planning allowance. Gross up for VAT/recoverability and add design, approvals, testing, contingency, escalation, temporary protection, parking/decant arrangements and financing carry. Keep separate conditional allowances for life-safety, drainage/waterproofing, structural discovery, energy controls/heating and possible authority contributions. Do not treat the Chemicon recommendation as committed expenditure.

### Value

The reported mechanical sensitivities of approximately EUR 13.0–18.1 million using annualised face rent and yields of 7.2%–5.2% are not a valuation. They exclude Opex, vacancy, CapEx, taxes, financing, lease expiry, tenant credit, specialist fit-out and planning constraints. Value should be based on verified NOI and lease duration, with explicit scenarios for public-occupier retention, expiry/break, reletting, current condition, garage CapEx, planning flexibility and exit yield. No residual or development value should be included before the planning and title conflicts are resolved.

### Financing

LTV, DSCR, refinance gap and net sale proceeds cannot be calculated from the current record. Require current debt and security evidence, verified NOI, value, CapEx timing, liquidity, insurance and lender consents. Run no-further-indexation, rent-dispute, vacancy/abatement, EUR 702,668–EUR 765,000 garage funding, broader remedial CapEx, +100 bp and +200 bp rate/refinance cases, stressed exit yield and maturity cases. Do not rely on uncommitted sponsor support or the nominal registered charge as evidence of benign debt.

### Exit

The strongest exit case is a documented, long-duration public-sector lease with lawful use, clean compliance, transferable maintenance records, funded CapEx and insurable risk. The downside is a specialised, security-heavy courthouse with a narrow buyer and tenant pool, high vacancy exposure on public-sector exit, constrained alternative use, unresolved parking/water rights and significant deferred CapEx. Exitability therefore depends more on legal and operational evidence than on Bad Homburg's general location quality. At exit, retain a complete lease abstract, current compliance pack, energy ledger, BMS and commissioning records, works warranties, insurance history, title/planning file and a funded transition roadmap.

### Property synthesis

# Final Property Decision — Amtsgericht Bad Homburg Courthouse Complex

## Decision outcome

**Proceed only conditionally. Do not approve unconditional acquisition, financing, valuation reliance or works release at this stage.**

The asset is credibly identified as the Amtsgericht Bad Homburg v.d. Höhe courthouse complex at Auf der Steinkaut 10–12, with a working physical baseline of parcels 122/11 and 123/1 totaling 6,361 m² and a recorded gross building volume of 24,499.12 m³. The court’s current operation at the address is independently corroborated by the official judiciary website. [citation:7ad1d5e5da65c9f2202cb2249b0386e7a603ffcf3551b1ab8021e944c3e20b56]

The positive case is real but incomplete:

- stated rent is EUR 78,255.17 per month from 1 January 2026, or EUR 939,062.04 annualised before costs, recoveries, vacancy, tax and debt service;
- the garage renewal has three close tender-stage prices between EUR 702,668.06 and EUR 719,950.93 net, against a EUR 765,000 net estimate;
- the site has active public-service use, public transport access and reported on-site and nearby parking; [citation:5b332ad6280d19f8250552918b4f35523b97704ca1a06229cfd358751edf6b73] [citation:11aaa9a6c87efc34f7e97c0aa0e1c509b55dfd5917780b149ffc0636689044a8] [citation:a30ef0682648649aa936bb24b366c6a862b83a42accceff7f2158b0d14c482f9]
- official records now establish that Bebauungsplan BHG_117 exists and is legally effective for the wider Gonzenheim area, although its parcel-specific application remains unknown. [citation:84df784cebb797bad789ff11f80085f2d5c9e0c3fd847ed48f4aa3ccf09ec85f] [citation:bc1977b4ec0f8301c8f709e8ac2c292d1ee3f401c2a11232fc8233ade758b9ff] [citation:8974c0da17169bc878d0780620a6f7135937ede7c9636c69fcdfb155eb78b7b1]

Those positives are outweighed, for commitment purposes, by unresolved hard gates:

1. certified title, current parcels, easements, Baulasten, planning overlay and release of the historic Eurohypo charge;
2. complete executed lease chain, tenant/payer identity, term, break rights, enforceable indexation, receipts, recoveries and security;
3. current structural, fire, HPPVO, escape-plan, drainage, fire-water, mechanical and electrical evidence;
4. executed garage works award, controlled scope, programme, funding, approvals, warranties and occupier protocol;
5. replacement maintenance cover because the FMC framework does not transfer on sale;
6. current debt, lender, maturity, covenants, liquidity and collateral evidence;
7. current insurance terms and asset-specific confirmation of flood, groundwater, fire, business-interruption and construction coverage; and
8. parcel- and works-specific water-protection, drainage, archaeology and hazard confirmations.

**Underwriting decision:** use only the documented face rent as a provisional gross-income input; assign no value to unverified recoveries, security, parking income, future indexation or residual/development upside; carry the garage works as a downside funding requirement; and preserve unquantified reserves for life-safety, drainage, water-protection, maintenance transition, energy transition, insurance and financing risks.

A material title or planning restriction, lease-form or rent dispute, current life-safety failure, unreleased security, unavailable insurance, materially broader works scope or inability to fund the hold-period obligations should trigger **repricing, escrow/indemnity, a works exclusion or rejection**.

## Asset and income performance

### Asset and operational performance

**Supported:** The property identity, courthouse use, current address, reported parcels, land area and building volume are established from the supplied property-file context. The official court page corroborates the judicial use and location. [citation:7ad1d5e5da65c9f2202cb2249b0386e7a603ffcf3551b1ab8021e944c3e20b56]

**Unknown:** Current physical condition and operational reliability are not established. The file contains a substantial inspection trail, but the reports are date-specific. The January 2026 status material records outdated escape/rescue plans, missing HPPVO reports and missing LBIH evidence of remediation. October and December 2025 fire-door and detector observations are evidence of observations at those dates, not proof that the defects remain open or have been closed.

The current technical review must therefore cover structure, envelope, waterproofing, garage, drainage, fire-water, fire doors, detection, escape plans, mechanical and electrical systems, CO systems, ventilation, water systems and emergency power. The absence of current structural documents and unreadable DWG material leaves structural capacity, as-built geometry, system capacity and approval status unresolved.

### Income performance

**Supported:** The stated monthly rent increased as follows:

| Effective period | Stated monthly rent | Annualised stated rent |
|---|---:|---:|
| 2023 | EUR 66,232.61 | EUR 794,791.32 |
| 2024 | EUR 72,458.48 | EUR 869,501.76 |
| From 2026 | EUR 78,255.17 | EUR 939,062.04 |

The recorded increases of 7.8%, 9.4% and 8.0% reconcile arithmetically to the preceding stated rents. The lease includes a 7.5% CPI threshold, but the base month, notice process, calculation convention, downward adjustments and 2025 history remain unverified. Destatis provides the official monthly and annual CPI series, but the lease’s reference month and notices are absent. [citation:37341105e8d2b06151c518663ce183229ce17c1857032ae876e5807f06372135] [citation:a36b3ef321bc61ccc3f42d0817d076b9f229b976c943e8e6994bdbaae5c2ccd9]

**Unknown:** EUR 939,062.04 is not established NOI. The file does not establish:

- whether the amount is net, gross or VAT-inclusive;
- service charges and recoveries;
- utilities, insurance, taxes and management costs;
- owner/tenant repair obligations;
- payment timing, arrears, credits or concessions;
- the leased area or EUR/m² rent;
- the legal tenant and paying entity;
- term, expiry, break or relocation rights;
- security, guarantee or budget commitment; or
- the complete signed lease and amendment chain.

Public courthouse occupation is credible, but it does not prove that the Amtsgericht, Land Hessen or another State entity is the contractual tenant or rent payer. Public occupation should not receive a State-backed credit premium until the executed lease, tenant certificate, remittance evidence, authority chain and payment history are verified.

### Market and value performance

**Inference:** Bad Homburg provides a supportive institutional and transport setting, but the specialised courthouse asset has materially narrower demand than ordinary office stock. Public market evidence reports 2024 average office rent of EUR 11.80/m², prime rent of EUR 17.60/m² and 20.2% vacancy. [citation:659c05bc82f8162c2cb6e2272ab329681f0793d7fc519f47c9328696c584cc1b] [citation:8bc3316e69868eb706415941b72766a27d40d491fd40588f142c29a1e1d2fa26]

The same market material reports weak 2024 office take-up of 9,100 m², 32% below 2023 and 49% below the prior ten-year average. [citation:8d44f2395d8e65c1188b2e1686b0c0911a6b5cc4cef9286fd9a05464bb30c134] This supports a location-positive but asset-specific demand conclusion, not a reliable reletting or exit assumption.

Mechanical capitalisations of the annualised face rent at 5.2%–7.2% produce approximately EUR 13.0–18.1 million, but these are not valuations. They exclude NOI leakage, vacancy, incentives, CapEx, taxes, lease duration, tenant credit, specialist fit-out, planning restrictions and financing. No residual or development value should be included until title and planning are resolved.

## Eight domain conclusions

### 1. Asset Integrity, Systems and Operational Resilience — **`unknown`**

The asset is identified and the garage tender evidence supports a current planning range, but present condition, compliance and operational continuity are not verified.

- The garage bids are EUR 702,668.06 net, EUR 704,397.29 net and EUR 719,950.93 net, compared with a EUR 765,000 net estimate.
- The Chemicon recommendation is tender-stage evidence only. The February 2026 ptd plan+ appointment is an LP 8 architectural/design-supervision appointment and does not prove a Chemicon award, construction start or completion.
- Retain EUR 765,000 net as the safer pre-award planning allowance, then add VAT/recoverability, design fees, testing, contingency, escalation, temporary protection, parking/decant costs and financing carry.
- Require current independent technical and life-safety inspection, a location-mapped defect register, HPPVO and LBIH records, current escape/rescue plans, fire-water tests, mechanical/electrical certificates, drainage evidence and completion documentation.
- Treat the garage programme as potentially disruptive to court operations until a method statement proves otherwise.

**Decision consequence:** Current technical and life-safety risk is high for evidence and closure; drainage and parking risk is medium; structural capacity remains unquantified.

### 2. Occupier, Lease, Income and Counterparty Economics — **`unknown`**

The face-rent trajectory is supported, but durable contractual income is not yet bankable.

- Underwrite EUR 78,255.17 per month only as unverified face rent.
- Treat recoveries, parking income, further indexation, security and public-counterparty support as zero or unknown until evidenced.
- Require the complete signed lease chain, schedules, side letters, notices, tenant certificate, payer identity, authority evidence, receipts, arrears statement, term, breaks, renewal rights, use/access obligations and works/abatement provisions.
- Reconcile each index adjustment to the contractual base month, official CPI observation, notice and receipt.
- Model a downside at the last undisputed rent, delayed indexation and potential credit or repayment claim.

**Decision consequence:** The asset may ultimately provide durable public-sector income, but current evidence supports only public occupation plus stated face rent, not a secured State-backed NOI stream.

### 3. Rights, Public Law and Ownership Governance — **`unknown`**

This is a hard acquisition and financing gate.

- The working parcel baseline is 122/11 and 123/1, but historic records also reference 122/9, 122/1 and 123/a. Current ownership, parcel continuity, Section II burdens, Section III charges, Baulasten, drainage rights and the Hesse easement remain unverified.
- Official records establish that Bebauungsplan BHG_117 exists, was resolved and publicly notified, and is legally effective. [citation:84df784cebb797bad789ff11f80085f2d5c9e0c3fd847ed48f4aa3ccf09ec85f] [citation:bc1977b4ec0f8301c8f709e8ac2c292d1ee3f401c2a11232fc8233ade758b9ff] [citation:8974c0da17169bc878d0780620a6f7135937ede7c9636c69fcdfb155eb78b7b1]
- The existence of the plan is now supported, but whether all or part of the current parcels fall within it, what designations apply, and what legal effect the listed supplement has remain unknown.
- The data-room statement that no Bebauungsplan applies and that §34 BauGB governs cannot be relied upon.
- The historic Eurohypo charge amounts are inconsistent by three orders of magnitude. Neither EUR 595,375,000 nor EUR 595,375 should be treated as current debt or as proof of release.
- The conditional Hesse easement may support courthouse use and parking but may constrain alternative use, vacant possession and exit.

**Decision consequence:** Obtain certified Grundbuch and Grundakten, current Baulastenverzeichnis, cadastral parcel history, executed BHG_117 plan and legally effective supplements, authenticated parcel overlay, easement deeds, charge-release documents and written planning confirmation before signing or closing.

### 4. Ground, Physical Climate and Insurability — **`unknown`**

The site is subject to material environmental and physical-risk interfaces, but no property-level loss probability or insurance position is established.

- Official location-level queries at the address points return HQS Bad Homburg qualitative Zone III and quantitative Zone C, and Pfingstborn drinking-water protection Zone IIIB. [citation:02a90f5dab3beec4c56173ba104fa481902b45a47e9ade9944a32d7b89b4cb] [citation:f14aeec3fd03cc5a153a2d01205cf9b3bc4b24a5afb372d07b8592c3e606cb76] [citation:e2498ef6a9db39caf07cb238e6cbcf6d240e2ab9fd5eb589d38fdc7174db69e1]
- Those results are strong location-level evidence, but the official dataset states that its delineations are for orientation and are not legally binding; the parcel-specific ordinance and authority position remain required. [citation:713e9cac2aef30b425a86814e7813e95fa87e7c06852b73868c7882ae8438a0]
- HWRM point queries at addresses 10 and 12 return `NoData` for HQ10, HQ100 and HQextrem. This is not evidence of zero flood risk and does not replace a building- and garage-footprint review. [citation:cf5815b5b5388efa1b13615df7e63c595017065d7268a0900fd1315f1ad4ab7b] [citation:039deef5ad3e475edde29686f8aab2a507f4f3885ad5132b298c66ab3721a3d0]
- Municipal Starkregen map values at the building and garage footprints have not been extracted.
- Historic records anticipate groundwater and require waterproof construction. Drainage calculations contain a 0.10 l/s inconsistency.
- ALTIS reported no current entries and no known contamination findings, and a 2006 certificate reported no UXO suspicion. These are dated or database findings, not intrusive or formal clearance.
- Current policy schedules, claims history, insurer survey, deductibles, exclusions and business-interruption coverage are absent.

**Decision consequence:** Require current water-authority confirmation, parcel protection-zone extract, Starkregen footprint readings, groundwater/drainage testing, archaeology and UXO responses, and binding insurance indications before intrusive works or reliance on clean insurability.

### 5. Energy, Carbon and Transition — **`unknown`**

The transition direction is material, but the baseline and compliance cost cannot yet be calculated.

- No 36-month energy series, fuel data, meter boundaries, energy certificate, plant schedule, capacity, BMS data, comfort data or emissions baseline is supplied.
- If qualifying plant exceeds 70 kW under the post-29 July 2026 regime, building automation and controls may be required by 31 December 2029, subject to exceptions. [citation:c001e507011b3402e5958eeda43750577c399c3e0bb9c605b44c2e39270b4638] [citation:f118c9bada245a7d8f5e6d176d1f0e2eedd3d82c336a0ed80c17317a8a1494e5]
- Heating obligations are date-controlled. For works through 28 July 2026, the relevant baseline is the pre-29 July 2026 GEG regime; for works from 29 July 2026, the later GModG regime applies, including escalating qualifying-fuel shares for new gas, oil or LPG systems. [citation:933c2c71d7fd7fe6fd0c403eb5fd6481f359779481872e5e6ad9d71478b9bbef] [citation:04bd3fd15a4fa1413008460c47a7bde855630326e711b98ed7e75b7d9528084f] [citation:2e848afac3e608f8cb48fcee76a72d69ab7edba58792f87146d8046c46f25f53]
- CO₂ cost allocation for qualifying non-residential leased space requires the landlord to bear at least 50%; the lease and fuel/meter arrangements must be checked. [citation:65b87dacf0f2cf162df015c65e223c11805d04f3ff9ac6c16eecf4512ed8e853]
- Bad Homburg’s heat-plan deadline is expected to be 30 June 2028 for a municipality of no more than 100,000 inhabitants, but completion and the resulting network economics are not established. [citation:f282dc4af15b03b812d8cd08b5dd2e587059293b05b3d469d2f3f33e18531621]

**Decision consequence:** Fund a first-stage metering and BMS audit; do not select a heating technology or approve like-for-like fossil replacement until plant data, works dates, legal regime, heat-plan status and water-protection feasibility are known.

### 6. Location, Demand, Market, Valuation and Exit — **`inference`**

The location supports institutional demand, but the asset’s specialist character, planning uncertainty and public-occupier concentration weaken exit certainty.

- The court’s active operation, public access, public transport and reported parking support continuity for public, regulated and professional users. [citation:5b332ad6280d19f8250552918b4f35523b97704ca1a06229cfd358751edf6b73] [citation:11aaa9a6c87efc34f7e97c0aa0e1c509b55dfd5917780b149ffc0636689044a8] [citation:a30ef0682648649aa936bb24b366c6a862b83a42accceff7f2158b0d14c482f9]
- General Bad Homburg market evidence cannot establish demand for a large, security-heavy courthouse with specialist fit-out, constrained parking and uncertain alternative use.
- No genuinely comparable courthouse sale, letting, buyer-depth or marketing-period evidence is supplied.
- If the public occupier exits, the downside may be a narrow tenant and buyer pool, extended vacancy, higher yield, major reletting works and constrained conversion value.
- No residual, development or alternative-use value should be underwritten before the BHG_117 overlay, easement and use permissions are confirmed.

**Decision consequence:** Obtain a specialist valuation based on verified NOI and lease duration, with explicit public-occupier retention, expiry/break, reletting, CapEx, planning and exit-yield scenarios.

### 7. Finance, Debt and Macro Transmission — **`unknown`**

Current financing cannot be analysed reliably.

- The historic Eurohypo reference identifies a lender-tracing route, not the current creditor, servicer, balance or discharge authority. Eurohypo was renamed Hypothekenbank Frankfurt AG, later rebranded as LSF, with main assets and liabilities transferred to Commerzbank; this does not establish the position for this specific loan. [citation:295c1def2f5a5099f0bf8e0b576e12e2103f6b034b475ac9f37d9219b4c859e6]
- No current principal, interest basis, maturity, amortisation, hedge, covenants, guarantees, recourse, liquidity or lender consent is evidenced.
- LTV, DSCR, covenant headroom, refinance gap and net sale proceeds cannot be calculated.
- The annualised face rent is not a debt-service measure.
- German and European evidence indicates tighter CRE lending and refinancing conditions, but those are market-level transmission indicators, not property-specific outcomes. [citation:937b3f0b01aa1427b1132f7857b027d1a7b5a0d8ad32ec93da9d762a6a52b794] [citation:30b0245b231fb2c97a5d10bfe982cc65993378a290aeac8f7f226d32d7448930]

**Decision consequence:** Require current lender certificates, loan and hedge documents, register/release evidence, valuation, 24-month liquidity forecast, capex funding plan and lender consents. Run no-further-indexation, rent-dispute, vacancy/abatement, garage-capex, broader-remediation, +100 bp and +200 bp rate/refinance cases.

### 8. External Dependencies, Geopolitics, Trade and Supply Chains — **`unknown`**

The immediate external dependency is maintenance and authority continuity rather than a demonstrated geopolitical loss.

- The FMC framework covers owner maintenance obligations but will not transfer on sale. Replacement procurement, mobilisation, service levels, open work orders, warranties and statutory-test coverage are unknown.
- The ptd LP 8 appointment creates a design and oversight dependency but does not evidence construction award or mobilisation.
- Current water-protection, drainage, municipal, fire and archaeology approvals are unresolved.
- The three garage bids support tender-stage price reasonableness, but not supply-chain resilience. Supplier countries, critical subcontractors, material origins, routes, lead times, buffers, sanctions screening, substitutions, force-majeure terms and insurance are unknown.
- No material geopolitical or trade pathway can be concluded from the available evidence; the risk should remain a contingent procurement and delivery risk rather than a fabricated probability.

**Decision consequence:** Make replacement maintenance, works award, supplier declarations, alternative specifications, lead-time evidence, authority consents, warranties, bonds and insurance completion conditions.

## Cross-domain transmission

### Legal identity → lawful works, collateral and value

Current parcels determine the correct title, easement, Baulasten, planning, water-zone and parking analysis. Those findings determine whether garage, drainage, fire-access, roof-space and energy works are lawful and financeable, and whether the lender receives the assumed collateral.

This chain is broken at the certified Grundbuch, cadastral history, current Baulasten, executed BHG_117 overlay and current rights records.

### Lease execution → income → NOI → debt service and value

The complete signed lease chain and tenant identity determine term, breaks, indexation, recoveries, obligations and remedies. Receipts establish effective income. Effective income then feeds NOI, DSCR, value, refinancing and exit.

This chain is broken by missing execution pages, schedules, notices, tenant certificate, receipts, arrears, recoveries, area schedule and security.

### Condition → continuity → insurance → financing

A current technical survey should produce a defect register, urgency ranking, scope, programme and cost. Fire, drainage and water-system close-out then supports lawful occupation, insurance and uninterrupted court operations.

The chain is broken by date-specific inspections, missing HPPVO/LBIH evidence, outdated plans, unreadable technical drawings and missing current structural, hydraulic and system records.

### Garage design → works completion → parking → occupier income

The ptd appointment should lead to a controlled specification, approvals, valid award, contractor mobilisation, phased access, testing and acceptance. Those steps determine parking availability, court disruption, potential abatement, claims and cost-to-complete.

The chain is broken between the LP 8 appointment and an evidenced construction award, and again at the missing programme, tenant protocol, approvals and funding.

### FMC expiry → maintenance Opex and operational resilience

Because the maintenance framework does not transfer, the buyer needs a replacement provider or valid novation, records, warranties, open-work-order schedule, statutory testing and emergency cover from completion.

The same maintenance-transition reserve should not be deducted separately as Opex, operational loss and compliance CapEx.

### Energy baseline → transition CapEx and exitability

Bills, meter boundaries and plant data establish the energy baseline. Plant capacity, age and controls determine automation and heating obligations. The local heat plan and water-protection regime determine feasible technology. The resulting pathway affects operating costs, tenant comfort, CapEx, carbon exposure and exit diligence.

This chain is broken by absent energy data, plant schedules, BMS information, floor area, heat-plan output and date-specific legal confirmation.

### Physical hazard → mitigation → insurance → lender conditions

Parcel-specific water, groundwater, Starkregen, radon, fire and other hazard evidence should feed mitigation design, insurer terms, deductibles, business-interruption cover and lender conditions.

The chain is broken by the absence of a current insurance schedule, claims history, insurer survey, binding hazard classification and footprint-level hazard interpretation.

### Debt and value → liquidity and exit proceeds

Verified debt and security terms combine with verified NOI, value and CapEx timing to determine LTV, DSCR, covenant headroom, refinancing gap and sale proceeds.

The chain is broken by the unresolved charge, missing loan records, absent funding plan, unverified rent and unquantified works.

## Contradictions and decision treatment

### Parcel identity

- **Reading A — supported working baseline:** parcels 122/11 and 123/1, totaling 6,361 m².
- **Reading B — unresolved legal history:** references also appear to parcels 122/9, 122/1 and 123/a.
- **Decision treatment:** use 122/11 and 123/1 only as a working physical scope. Do not treat them as title proof. The certified Grundbuch, cadastral map and Fortführungs records must establish whether the courthouse, access, parking and drainage are acquired.

### Planning regime

- **Reading A:** the property-file material states an Innenbereich under §34 BauGB and no Bebauungsplan.
- **Reading B — now supported at area level:** official records establish legally effective Bebauungsplan BHG_117, including Auf der Steinkaut in its title. [citation:84df784cebb797bad789ff11f80085f2d5c9e0c3fd847ed48f4aa3ccf09ec85f] [citation:6fd29480cd1d3547f671025eb36f430ec031223e887640d0c8eb4bac09071bf2]
- **Decision treatment:** the existence of BHG_117 is resolved, but parcel applicability, designation, building lines, parking/access controls and supplement effect are not. Exclude expansion, conversion and residual value until the executed plan and parcel overlay are confirmed.

### Water-protection designations

- **Reading A:** HQS Bad Homburg healing-spring protection.
- **Reading B:** Pfingstborn drinking-water protection, with earlier records referring inconsistently to Zone IIIA, IIIB and other descriptions.
- **Decision treatment:** these may be overlapping designations rather than a direct contradiction. Official address-point evidence supports HQS qualitative III/quantitative C and Pfingstborn IIIB, but the state dataset is orientation-only. [citation:02a90f5dab3beec4c56173ba104fa481902b45a47e9ade9944a32d7b89b4cb] [citation:f14aeec3fd03cc5a153a2d01205cf9b3bc4b24a5afb372d07b8592c3e606cb76] [citation:e2498ef6a9db39caf07cb238e6cbcf6d240e2ab9fd5eb589d38fdc7174db69e1] Obtain the legally applicable ordinances, parcel extract and work-specific authority position.

### Parking

- **Reading A:** 1988 material records 89 spaces, approximately 55 underground and 34 outdoor.
- **Reading B:** 1985 material refers to 89 required spaces but describes 58 underground plus 39 visitor spaces, totaling 97; historic compensation records refer to DM 213,000 before a stated cap and DM 200,000 for ten spaces.
- **Decision treatment:** the later arithmetic is coherent but not proven operative. Confirm current approved count, accessible and fire-access spaces, easement scope, compensation payment and garage-work phasing. Do not recognize a current shortfall or parking income, but do not assume the issue is closed.

### Drainage

- **Reading A:** historic total rainwater flow of 33.52 l/s.
- **Reading B:** another page gives 33.62 l/s and references a drainage-canal easement linked to historic parcel 122/9.
- **Decision treatment:** the discrepancy does not establish inadequacy, but prevents reliance on the historic calculation. Obtain current hydraulic calculations, connection approvals, capacity, backflow protection, easement access and maintenance responsibility.

### Registered charge and debt

- **Reading A:** EUR 595,375,000 nominal Eurohypo charge.
- **Reading B:** EUR 595,375 release-entry amount.
- **Decision treatment:** neither amount is current debt evidence. Possibilities include transcription error, nominal security, release/assignment issue or cross-collateralisation. Require current register, notarial instruments and authenticated lender/servicer confirmation.

### Fire safety

- **Reading A:** historic fire-water requirement of 2,400 l/min for two hours, inspection intervals and dated observations.
- **Reading B:** current status is unresolved because HPPVO reports, updated escape plans and LBIH closure evidence are missing.
- **Decision treatment:** dated observations are not proof of current failure, but remediation assertions are not proof of closure. Require current tests, certificates, plans and authority acceptance.

### Garage procurement

- **Reading A:** three close bids indicate a competitive tender range.
- **Reading B:** no construction award is evidenced; the ptd appointment concerns architectural LP 8 services.
- **Decision treatment:** use EUR 702,668.06–EUR 765,000 net as planning/downside evidence only. Do not treat Chemicon as appointed or the amount as committed until the executed contract, scope, programme, funding and warranties are supplied.

### Public occupation and contractual income

- **Reading A:** official court records confirm active courthouse use at the address. [citation:7ad1d5e5da65c9f2202cb2249b0386e7a603ffcf3551b1ab8021e944c3e20b56]
- **Reading B:** legal tenant, payer, guarantee, appropriation, term and payment history are unknown.
- **Decision treatment:** treat public occupation as operational support, not contractual State credit or long-duration income evidence.

### Lease-form analysis

- **Reading A:** the 2020 amendment requires handwritten signatures by both parties on the same instrument.
- **Reading B:** statutory form and transition rules differ by lease and amendment dates; the contractual covenant may be stricter than the current statutory minimum.
- **Decision treatment:** do not conclude invalidity or termination. Require the complete signed chronology, incorporated schedules, side letters and date-specific legal-form opinion.

### No-findings evidence versus clearance

- **Reading A:** ALTIS reported no current entries/no known contamination findings; UXO material reported no suspicion.
- **Reading B:** these are dated or database findings, not intrusive clearance, current groundwater evidence or warranties.
- **Decision treatment:** narrow the risk from demonstrated contamination or loss to unverified residual exposure. Do not convert no-findings into a positive all-clear.

### Heating-law regime

- **Reading A:** pre-29 July 2026 GEG transition rules.
- **Reading B:** post-29 July 2026 GModG rules with different fossil-fuel and automation provisions.
- **Decision treatment:** the applicable branch depends on installation and commissioning dates. Preserve both legal and financial cases until those dates and the governing statutory version are confirmed.

## Evidence dependence and agreement limits

All researchers used the **same model and shared runtime** and relied substantially on the same supplied property-file context. Agreement among the reports is therefore **not independent confirmation, not field-wide consensus and not multiple-source corroboration**.

In particular:

- repeated rent figures are one lease-chain evidence stream, not independent income confirmation;
- the three garage bids are same-scope tender evidence, not three independent market transactions;
- repeated planning concerns do not resolve the plan conflict;
- repeated fire-safety references do not prove current failure or closure;
- the official court page corroborates location and use, not lease liability, payer, security or duration;
- official water and hazard sources establish designated regimes or regional screening, not parcel-level loss probability or work permission; and
- Commerzbank’s historic lender-chain evidence provides a tracing route, not proof of the current lender, debt balance or release.

## Assert / caveat / avoid guidance

### Assert

- The asset is credibly the active Amtsgericht Bad Homburg courthouse complex at Auf der Steinkaut 10–12.
- The current stated face rent is EUR 78,255.17 per month, equivalent to EUR 939,062.04 annualised before deductions.
- The face-rent trajectory is documented, but normalized NOI is not.
- BHG_117 is an established, legally effective plan record for the wider area; parcel-specific application remains unresolved. [citation:84df784cebb797bad789ff11f80085f2d5c9e0c3fd847ed48f4aa3ccf09ec85f]
- The garage tender range is approximately EUR 0.70–0.72 million net, with EUR 765,000 net the safer pre-award planning allowance.
- The FMC maintenance framework is reported not to transfer on sale.
- The site is subject to location-level healing-spring and drinking-water protection designations, requiring work-specific authority review. [citation:02a90f5dab3beec4c56173ba104fa481902b45a47e9ade9944a32d7b89b4cb] [citation:f14aeec3fd03cc5a153a2d01205cf9b3bc4b24a5afb372d07b8592c3e606cb76] [citation:e2498ef6a9db39caf07cb238e6cbcf6d240e2ab9fd5eb589d38fdc7174db69e1]
- HWRM `NoData` at the two queried address points is not a zero-flood-risk conclusion. [citation:cf5815b5b5388efa1b13615df7e63c595017065d7268a0900fd1315f1ad4ab7b] [citation:039deef5ad3e475edde29686f8aab2a507f4f3885ad5132b298c66ab3721a3d0]

### Caveat

- Face rent is not NOI and public occupation is not a State guarantee.
- Historic inspections and dated defect observations do not establish present condition.
- Remediation assertions without HPPVO/LBIH evidence do not establish closure.
- No current debt or financing conclusion can be drawn from a nominal Grundschuld.
- The Chemicon recommendation is not an executed works award.
- No current insurance recovery should be assumed.
- No-finding contamination, UXO or archaeology records are not formal or intrusive clearances.
- Future indexation is contractual and notice-dependent, not an automatic CPI assumption.
- Energy transition cost and stranding risk remain unknown until consumption, plant, works dates and legal regime are verified.
- Market rents, yields and citywide hazard events are contextual evidence, not subject-property comparables.

### Avoid

- Do not approve an unconditional acquisition or financing gate.
- Do not capitalize EUR 939,062.04 as NOI.
- Do not assign a public-counterparty premium, long lease duration or renewal probability without the executed lease and receipts.
- Do not treat BHG_117’s existence as proof of parcel-specific restrictions, or the old §34 statement as proof that no plan applies.
- Do not assume the Eurohypo charge was released or harmless.
- Do not assume current parking compliance, drainage capacity, fire-water availability or lawful alternative use.
- Do not treat the ptd appointment as proof that Chemicon was appointed or that garage works are funded.
- Do not release the garage reserve merely because the bids are close or the tender adviser reported no technical objections.
- Do not double-count garage CapEx, disruption, fire-safety, water, maintenance, insurance or energy risks across multiple valuation deductions.
- Do not infer a clean site, zero flood risk, available insurance or low geopolitical exposure from missing adverse evidence.